# Software profesional en Acústica 2024-25 (M2i)

*This notebook contains a modification of the notebook [FEM_Helmholtz_equation_Robin](https://github.com/spatialaudio/computational_acoustics/blob/master/FEM_Helmholtz_equation_Robin.ipynb), created by Sascha Spors, Frank Schultz, Computational Acoustics Examples, 2018. The text/images are licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/). The code is released under the [MIT license](https://opensource.org/licenses/MIT).*

First, we need to install on the fly [NGSolve](https://ngsolve.org/) using the [FEM on Colab](https://fem-on-colab.github.io/packages.html) install script:

In [1]:
try:
    import ngsolve
except ImportError:
    !wget "https://fem-on-colab.github.io/releases/ngsolve-install-release-complex.sh" -O "/tmp/ngsolve-install.sh" && bash "/tmp/ngsolve-install.sh"
    import ngsolve

# Numerical Solution of the Helmholtz Equation with Robin Boundary Conditions using the Finite Element Method

This notebook illustrates the numerical solution of the wave equation for an harmonic excitation and Robin boundary conditions using the [Finite Element Method](https://en.wikipedia.org/wiki/Finite_element_method) (FEM). The method aims at an approximate solution by subdividing the area of interest into smaller parts with simpler geometry, linking these parts together and applying methods from the calculus of variations to solve the problem numerically. The FEM is a well established method for the numerical approximation of the solution of partial differential equations (PDEs). The solutions of PDEs are often known analytically only for rather simple geometries. FEM based simulations allow to gain insights into other more complex cases.

## Problem Statement

The inhomogeneous [Helmholtz equation](https://en.wikipedia.org/wiki/Helmholtz_equation) is given as

\begin{equation*}
\Delta P(\boldsymbol{x}, \omega) + \frac{\omega^2}{c^2} P(\boldsymbol{x}, \omega) = - F(\boldsymbol{x}, \omega) .
\end{equation*}

We aim for a numerical solution of the Helmholtz equation on the domain $\Omega$ with respect to the homogeneous Robin boundary condition

\begin{equation*}
V_n(\boldsymbol{x}, \omega) + \frac{1}{Z(\boldsymbol{x}, \omega)} P(\boldsymbol{x}, \omega) = 0 \qquad \text{for } \boldsymbol{x} \in \partial \Omega,
\end{equation*}

where $V_n(\boldsymbol{x}, \omega)$ denotes the particle velocity in inward normal direction to the boundary $\partial \Omega$ of $\Omega$ and $Z(\boldsymbol{x}, \omega)$ the surface impedance of the boundary.
The particle velocity can be linked to the pressure using the Euler equation

\begin{equation*}
-\mathrm{j} \omega \rho_0 V_n(\boldsymbol{x}, \omega) = \frac{\partial}{\partial n} P(\boldsymbol{x}, \omega) ,
\end{equation*}

where $\rho_0$ denotes the mass density of air.
Introducing this into the Robin boundary equation above yields

\begin{equation*}
\frac{\partial}{\partial n} P(\boldsymbol{x}, \omega) - \mathrm{j} \underbrace{\frac{\omega \rho_0}{Z}}_{\sigma} P(\boldsymbol{x}, \omega) = 0 \qquad \text{for } \boldsymbol{x} \in \partial \Omega .
\end{equation*}

The medium impedance of air for free-field propagation is $Z_0 = \rho_0 c$, hence $\sigma_0 = \frac{\omega}{c}$ in this case. Free-field conditions can be simulated by matching the impedance of the boundary to $Z_0$.

## Variational Formulation

Starting from the [variational formulation of the Helmholtz equation](FEM_Helmholtz_equation_2D.ipynb#Variational-Formulation) (before application of Green's first theorem)

\begin{equation*}
{-} \int_\Omega \nabla P(\boldsymbol{x}, \omega) \cdot \nabla Q(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{x}  + \int_{\partial \Omega} Q(\boldsymbol{x}, \omega) \frac{\partial}{\partial n}  P(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{s} + \frac{\omega^2}{c^2} \int_\Omega P(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{x} = -\int_\Omega F(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{x}
\end{equation*}

and introducing the Robin boundary condition into the second integral yields

\begin{equation*}
{-} \int_\Omega \nabla P(\boldsymbol{x}, \omega) \cdot \nabla Q(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{x}  + \mathrm{j} \sigma \int_{\partial \Omega} Q(\boldsymbol{x}, \omega) P(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{s} + \frac{\omega^2}{c^2} \int_\Omega P(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{x} = -\int_\Omega F(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{x} .
\end{equation*}

It is common to express this integral equation in terms of the bilinear $a(P, Q)$ and linear $L(Q)$ forms 

\begin{equation*}
a(P, V) = \frac{\omega^2}{c^2} \int_\Omega P(\boldsymbol{x}, \omega) Q(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{x} - \int_\Omega \nabla P(\boldsymbol{x}, \omega) \cdot \nabla Q(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{x} + \mathrm{j} \sigma \int_{\partial \Omega} Q(\boldsymbol{x}, \omega) P(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{s} , \end{equation*}

\begin{equation*}
L(V) = -\int_\Omega Q(\boldsymbol{x}, \omega) F(\boldsymbol{x}, \omega)\, \mathrm{d}\boldsymbol{x} ,
\end{equation*}

where

\begin{equation*}
a(P, Q) = L(Q) \qquad \forall Q.
\end{equation*}

## Numerical Solution

The numerical solution of the variational problem is based on [FEniCS](https://fenicsproject.org/), an open-source framework for numerical solution of PDEs.
Its high-level Python interface `dolfin` is used in the following to define the problem and compute the solution.
The implementation is based on the variational formulation derived above.
It is common in the FEM to denote the solution of the problem by $u$ and the test function by $v$.
The definition of the problem in FEniCS is very close to the mathematical formulation of the problem.

For the subsequent examples the solution of inhomogeneous wave equation for a point source $F(\boldsymbol{x}) = \delta(\boldsymbol{x}-\boldsymbol{x}_s)$ at position $\boldsymbol{x}_s$ is computed using the FEM.
A function is defined for this purpose, as well as for the plotting of the resulting sound field.

In [2]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw

def FEM_Helmholtz(mesh, frequency, xs, sigma, c=343):
    """Numerical solution of the Helmholtz equation using NGSolve.

    Uses complex-valued H1 space so the formulation is fully complex
    even when the problem data are real.

    Parameters
    ----------
    mesh        : ngsolve.Mesh
    frequency   : float, excitation frequency in Hz
    xs          : (float, float), point-source position (x, y)
    sigma       : float, Robin parameter (Pa·s/m)
    c           : float, speed of sound (m/s)

    Returns
    -------
    u : ngsolve.GridFunction (complex)
    """
    # squared wavenumber
    k2 = (2 * np.pi * frequency / c) ** 2

    # ── function space: CG order 2, complex-valued ─────────────────────────
    V = H1(mesh, order=2, complex=True)

    u = V.TrialFunction()
    v = V.TestFunction()

    # ── bilinear form  a(u,v) = (∇u,∇v) - k² (u,v)  ─────────────────────────
    a = BilinearForm(V)
    a += grad(u) * grad(v) * dx - k2 * u * v * dx - 1j * sigma * u * v * ds
    a.Assemble()

    # ── linear form  L(v) = Dirac's delta at x=xs  ─────────────────────────
    xp, yp = xs
    f = LinearForm(V)
    f += v(xp, yp)
    f.Assemble()

    # Build solution GridFunction
    sol = GridFunction(V)

    # ── solve ───────────────────────────────────────────────────────────────
    sol.vec.data = a.mat.Inverse(V.FreeDofs()) * f.vec

    return sol

### Sound Field in a Rectangular Room

The two-dimensional sound field in a rectangular room (rectangular plate) with homogeneous Robin boundary conditions is computed for the frequency $f=120$ Hz and source position $x_s = (1.2,3.2)$ m.

In [3]:
f = 120  # frequency
xs = (1.2, 3.2)  # source position

Generate the mesh

In [4]:
from netgen.geom2d import SplineGeometry

def make_rect_mesh(x0, y0, x1, y1, maxh=0.1):
    """Create a rectangular NGSolve mesh from (x0,y0) to (x1,y1)."""
    geo = SplineGeometry()
    p1 = geo.AppendPoint(x0, y0)
    p2 = geo.AppendPoint(x1, y0)
    p3 = geo.AppendPoint(x1, y1)
    p4 = geo.AppendPoint(x0, y1)
    geo.Append(["line", p1, p2], bc="bottom")
    geo.Append(["line", p2, p3], bc="right")
    geo.Append(["line", p3, p4], bc="top")
    geo.Append(["line", p4, p1], bc="left")

    # Generate mesh
    ngmesh = geo.GenerateMesh(maxh=maxh)
    return Mesh(ngmesh)

# define geometry and mesh  (5 m × 4 m room, fine mesh ≈ mshr resolution 200)
mesh = make_rect_mesh(0, 0, 5, 4, maxh=0.1)

# Plot mesh
Draw(mesh, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

BaseWebGuiScene

First, the case of sound-hard (Neumann) boundary conditions with $\sigma = 0$ ($Z \to \infty$) is considered.

In [5]:
# compute solution  (Neumann → sound-hard walls)
gfu = FEM_Helmholtz(mesh, 120, xs, sigma=0)

# plot the modulus of the pressure field
Draw(sqrt(gfu.real**2 + gfu.imag**2), mesh, height="3vh")

# plot the the pressure field animated with the phase values
Draw(gfu, mesh, "solution", animate_complex=True, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

BaseWebGuiScene

Now the case of a matched boundary (free-field propagation) with $\sigma = \frac{\omega}{c}$ ($Z = Z_0$) is considered.

In [6]:
# compute solution for free-field propogation
gfu = FEM_Helmholtz(mesh, 120, xs, sigma=2*np.pi*f/343)

# plot the modulus of the pressure field
Draw(sqrt(gfu.real**2 + gfu.imag**2), mesh, height="3vh")

# plot the the pressure field animated with the phase values
Draw(gfu, mesh, "solution", animate_complex=True, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

BaseWebGuiScene

The last example shows the simulation result for sound-soft (Dirichlet) boundary conditions with $\sigma \to \infty$ ($Z=0$).

In [7]:
# compute solution for very large sigma
gfu = FEM_Helmholtz(mesh, 120, xs, sigma=1e5)

# plot the modulus of the pressure field
Draw(sqrt(gfu.real**2 + gfu.imag**2), mesh, height="3vh")

# plot the the pressure field animated with the phase values
Draw(gfu, mesh, "solution", animate_complex=True, height="3vh")

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.260…

WebGuiWidget(layout=Layout(height='3vh', width='100%'), value={'gui_settings': {'Complex': {'phase': 0.0, 'spe…

BaseWebGuiScene

The effect of the homogeneous Dirichlet boundary condition (zero pressure at the boundary) can be observed conveniently by inspecting the magnitude of the sound field close to the boundary.

**Copyright**

This notebook is provided as [Open Educational Resource](https://en.wikipedia.org/wiki/Open_educational_resources). Feel free to use the notebook for your own purposes. The text is licensed under [Creative Commons Attribution 4.0](https://creativecommons.org/licenses/by/4.0/), the code of the IPython examples under the [MIT license](https://opensource.org/licenses/MIT).